In [1]:
# 导入必要的库
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFECV, SelectFromModel
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import json
import shap
import matplotlib.pyplot as plt
from functools import partial
import warnings
warnings.simplefilter("ignore")

# ====================== 辅助函数定义 ======================
def get_shap_best_features(model, X_train, is_tree=True):
    """使用SHAP计算特征重要性"""
    try:
        if is_tree:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_train)
            if isinstance(shap_values, list) and len(shap_values) == 2:
                shap_values = shap_values[1]
        else:
            explainer = shap.KernelExplainer(
                model.predict_proba, 
                shap.sample(X_train, 5, random_state=123)  # 添加 random_state
            )
            shap_values = explainer.shap_values(X_train)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]
            else:
                shap_values = shap_values[:, :, 1]
        
        if len(shap_values.shape) == 3:
            shap_values = shap_values[0]
            
        shap_sum = np.abs(shap_values).mean(axis=0).flatten()
        importance_df = pd.DataFrame({
            'column_name': X_train.columns.tolist(),
            'shap_importance': shap_sum
        })
        return importance_df.sort_values('shap_importance', ascending=False)
    except Exception as e:
        print(f"SHAP计算错误: {str(e)}")
        return pd.DataFrame()

def get_SelectFromModel_features(X, y, model):
    """基于模型重要性筛选特征"""
    try:
        sel = SelectFromModel(model)
        sel.fit(X, y)
        return X.columns[sel.get_support()].tolist()
    except Exception as e:
        print(f"SelectFromModel错误: {str(e)}")
        return []

def get_RFECV_features(X, y, model, nfold=5):
    """递归特征消除"""
    try:
        rfecv = RFECV(
            estimator=model,
            step=1,
            cv=StratifiedKFold(nfold, shuffle=True, random_state=123),
            scoring='roc_auc'
        )
        rfecv.fit(X, y)
        print(f"RFECV选择特征数: {rfecv.n_features_}")
        return X.columns[rfecv.get_support()].tolist()
    except Exception as e:
        print(f"RFECV错误: {str(e)}")
        return []

def get_k_best_features(X, y, k=10, score_func=mutual_info_classif, **func_kwargs):
    """基于统计检验筛选特征"""
    try:
        # 将参数绑定到 score_func
        bound_score_func = partial(score_func, **func_kwargs)
        selector = SelectKBest(score_func=bound_score_func, k=k)
        selector.fit(X, y)
        return X.columns[selector.get_support()].tolist()
    except Exception as e:
        print(f"KBest错误: {str(e)}")
        return []

def get_ensemble_features(*feature_lists):
    """集成特征选择结果"""
    try:
        all_features = list(set().union(*feature_lists))
        feature_counter = Counter([f for sublist in feature_lists for f in sublist])
        return {
            'all_features': sorted(all_features),
            '0.75_features': [f for f, cnt in feature_counter.items() if cnt/len(feature_lists) >= 0.75],
            '0.5_features': [f for f, cnt in feature_counter.items() if cnt/len(feature_lists) >= 0.5]
        }
    except Exception as e:
        print(f"特征集成错误: {str(e)}")
        return {}


# ====================== 新增全局预处理函数 ======================
def create_preprocessor(features, continuous_cols, categorical_cols):
    """动态创建预处理管道（全局可访问）"""
    current_cont = [f for f in features if f in continuous_cols]
    current_cat = [f for f in features if f in categorical_cols]
    return ColumnTransformer([
        ('num', StandardScaler(), current_cont),
        ('cat', 'passthrough', current_cat)
    ], remainder='drop')
    
    
    
    
    
# ====================== 改进后的SHAP可视化部分 ======================
def generate_shap_plots(model_name, features, cont_cols, cat_cols):
    """通用SHAP可视化生成函数"""
    try:
        print(f"\n生成 {model_name} 的SHAP可视化...")
        
        # 创建预处理和模型管道
        preprocessor = create_preprocessor(features, cont_cols, cat_cols)
        
        if model_name == 'RandomForest':
            model = RandomForestClassifier(n_estimators=100, random_state=123)
            explainer_type = 'tree'
        elif model_name == 'LogisticRegression':
            model = LogisticRegression(max_iter=1000, class_weight='balanced')
            explainer_type = 'linear'
        else:
            return

        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        # 训练模型
        pipeline.fit(X_train[features], y_train)
        
        # 准备解释数据
        X_processed = pipeline.named_steps['preprocessor'].transform(X_train[features])
        background = shap.sample(X_processed, 100,random_state=123)  # 使用采样加速计算

        # 创建解释器
        if explainer_type == 'tree':
            explainer = shap.TreeExplainer(pipeline.named_steps['classifier'])
            shap_values = explainer.shap_values(X_processed)
            if isinstance(shap_values, list):
                shap_values = shap_values[1]  # 取正类的SHAP值
        else:
            explainer = shap.KernelExplainer(
                lambda x: pipeline.predict_proba(x)[:,1], 
                background
            )
            shap_values = explainer.shap_values(X_processed, nsamples=50)
        
        # 可视化设置
        plt.figure(figsize=(12, 8))
        
        # 蜂窝点状图
        shap.summary_plot(
            shap_values, 
            X_processed,
            feature_names=features,
            plot_type="dot",  # 改为点状图
            show=False,
            color=plt.get_cmap("coolwarm"),
            plot_size=(12, 8)
        )
        
        # 添加临床相关标注
        plt.title(f'{model_name} SHAP Feature Importance\n(AUC={test_auc_dict.get(model_name, "N/A")})', 
                fontsize=14, pad=20)
        plt.xlabel('SHAP Value (Impact on Mortality Risk)', fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        
        # 突出显示关键临床特征
        ax = plt.gca()
        yticks = ax.get_yticklabels()
        for label in yticks:
            if label.get_text() in ['sofa', 'apsiii', 'Charlson']:
                label.set_color('darkred')
                label.set_fontweight('bold')
        
        plt.tight_layout()
        plt.savefig(f'shap_{model_name}.png', dpi=300, bbox_inches='tight')
        plt.close()
        print(f"{model_name} SHAP图已保存为shap_{model_name_0524}_3.png")
        
    except Exception as e:
        print(f"{model_name} SHAP可视化失败: {str(e)}")

In [ ]:
# ====================== 主流程函数 ======================
def load_and_preprocess():
    """数据加载与预处理"""
    data = pd.read_csv(r"E:\100-科研\400_镁\006重启版\1.7_2.1.csv")
  
    # 定义常量和特征列
    IND_COLS = ['death_hosp', 'death_icu', 'death_ICU90days']
    LABEL = "death_ICU28days"
    exclude_cols = IND_COLS + [LABEL]
    
    continuous_cols = [
        'Age', 'Weight', 'WBC', 'RBC', 'Platelets', 'Hemoglobin', 
        'Glucose', 'Magnesium', 'Potassium', 'Sodium', 'Calcium',
        'Chloride', 'Phosphate', 'BUN',  'Creatinine','Charlson','sofa'
    ]
    
    categorical_cols = [
        'gender', 'diabetes','cerebrovascular_disease','congestive_heart_failure', 'renal_disease',
        'malignant_cancer','liver_disease','chronic_pulmonary_disease',
        'peripheral_vascular_disease', 'myocardial_infarct', 'hypertension','ventilation',
         'rrt', 'potassium_supplementation',
        'calcium_supplementation', 'magnesium_sulfate_supplementation'
    ]
    
    # 划分数据集
    X = data[continuous_cols + categorical_cols]
    y = data[LABEL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=123
    )
    
    return X_train, X_test, y_train, y_test, continuous_cols, categorical_cols

def feature_selection_pipeline(X_train, y_train, continuous_cols, categorical_cols):
    """特征选择主流程"""
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), continuous_cols),
        ('cat', 'passthrough', categorical_cols)
    ])
    
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=123),
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=123),
        'KNN': KNeighborsClassifier()
    }
    
    results = {}
    
    for model_name, model in models.items():
        print(f"\n=== 正在处理 {model_name} ===")
        model_results = {}
        
        try:
            # 数据预处理
            X_processed = preprocessor.fit_transform(X_train)
            X_processed = pd.DataFrame(
                X_processed, 
                columns=continuous_cols + categorical_cols
            )
            
            # 模型训练
            model.fit(X_processed, y_train)
            
            # 特征选择方法
            feature_methods = {}
            
            # RFECV（跳过KNN）
            if model_name != 'KNN':
                print("运行RFECV...")
                rfecv_features = get_RFECV_features(X_processed, y_train, model)
                feature_methods['RFECV'] = rfecv_features
                
                print("运行SelectFromModel...")
                sfm_features = get_SelectFromModel_features(X_processed, y_train, model)
                feature_methods['SelectFromModel'] = sfm_features
            
            # KBest
            print("运行KBest...")
            mi_with_seed = partial(mutual_info_classif, random_state=123)

           # 对于每个模型：
            k_best = get_k_best_features(
            X_processed, y_train, 
            k=20,
            score_func=mi_with_seed
            )
            feature_methods['KBest'] = k_best
            
            # SHAP（仅随机森林）
            if model_name == 'RandomForest':
                print("计算SHAP值...")
                shap_df = get_shap_best_features(model, X_processed)
                if not shap_df.empty:
                    shap_features = shap_df.head(20)['column_name'].tolist()
                    feature_methods['SHAP'] = shap_features
            
            # 集成特征
            ensemble_features = get_ensemble_features(*feature_methods.values())
            results[model_name] = {**feature_methods, **ensemble_features}
            
        except Exception as e:
            print(f"{model_name} 处理失败: {str(e)}")
            continue
    
    # 保存结果
    with open("feature_selection_results0420_0524.json", "w") as f:
        json.dump(results, f, indent=2)
    
    return results


# 修改后的test_set_evaluation函数
def test_set_evaluation(feature_results, X_train, X_test, y_train, y_test, continuous_cols, categorical_cols):
    """测试集评估"""
    def create_preprocessor(features):
        current_cont = [f for f in features if f in continuous_cols]
        current_cat = [f for f in features if f in categorical_cols]
        return ColumnTransformer([
            ('num', StandardScaler(), current_cont),
            ('cat', 'passthrough', current_cat)
        ], remainder='drop')
    
    def print_markdown(df):
        """兼容性打印函数"""
        headers = df.columns.tolist()
        data = df.values.tolist()
        header = "| " + " | ".join(headers) + " |"
        separator = "|-" + "-|-".join(["-"*len(h) for h in headers]) + "-|"
        rows = ["| " + " | ".join(map(str, row)) + " |" for row in data]
        print("\n".join([header, separator] + rows))
    
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=123),
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=123),
        'KNN': KNeighborsClassifier()
    }
    
    results = []
    
    for model_name in models:
        print(f"\n=== 评估 {model_name} ===")
        try:
            # 获取特征集
            if model_name not in feature_results:
                print(f"{model_name} 无有效特征结果")
                continue
                
            features = feature_results[model_name].get('0.75_features', [])
            if not features:
                print(f"使用all_features替代空特征集")
                features = feature_results[model_name].get('all_features', [])
            
            # 创建处理管道
            preprocessor = create_preprocessor(features)
            pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', models[model_name])
            ])
            
            # 训练与评估
            pipeline.fit(X_train[features], y_train)
            y_proba = pipeline.predict_proba(X_test[features])[:,1]
            auc = roc_auc_score(y_test, y_proba)
            
            results.append({
                'Model': model_name,
                'AUC': round(auc, 3),
                'Features': len(features),
                'Feature_List': features
            })
            
        except Exception as e:
            print(f"{model_name} 评估失败: {str(e)}")
    
    # 输出结果
    print("\n=== 测试集表现 ===")
    if results:
        results_df = pd.DataFrame(results)
        print_markdown(results_df[['Model', 'AUC', 'Features']])
        results_df.to_csv("test_performance.csv", index=False)
    else:
        print("无有效评估结果")
    return results
# ====================== 主执行流程 ======================
if __name__ == "__main__":
    # 数据加载
    X_train, X_test, y_train, y_test, cont_cols, cat_cols = load_and_preprocess()
    
    # 特征选择
    print("\n" + "="*30 + " 开始特征选择 " + "="*30)
    feature_results = feature_selection_pipeline(X_train, y_train, cont_cols, cat_cols)
    
    # 测试评估
    print("\n" + "="*30 + " 开始测试评估 " + "="*30)
    test_results = test_set_evaluation(feature_results, X_train, X_test, y_train, y_test, cont_cols, cat_cols)
    
    # SHAP可视化
    test_auc_dict = {res['Model']: res['AUC'] for res in test_results} if test_results else {}
    
    # 为随机森林和逻辑回归生成SHAP图
    for model_name in ['RandomForest', 'LogisticRegression']:
        if model_name in feature_results:
            features = feature_results[model_name].get('0.75_features', [])
            if len(features) > 0:
                generate_shap_plots(model_name, features, cont_cols, cat_cols)


============================== 开始特征选择 ==============================

=== 正在处理 RandomForest ===
运行RFECV...
RFECV选择特征数: 33
运行SelectFromModel...
运行KBest...
计算SHAP值...

=== 正在处理 LogisticRegression ===
运行RFECV...
RFECV选择特征数: 30
运行SelectFromModel...
运行KBest...

=== 正在处理 KNN ===
运行KBest...

============================== 开始测试评估 ==============================

=== 评估 RandomForest ===

=== 评估 LogisticRegression ===

=== 评估 KNN ===

=== 测试集表现 ===
| Model | AUC | Features |
|-------|-----|----------|
| RandomForest | 0.837 | 19 |
| LogisticRegression | 0.825 | 8 |
| KNN | 0.713 | 20 |

生成 RandomForest 的SHAP可视化...


In [1]:
# -*- coding: utf-8 -*-
"""
独立评估流程（需先运行特征选择生成feature_selection_results.json）
"""
import json
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
# ====================== 配置参数 ======================
DATA_PATH = r"E:\100-科研\400_镁\006重启版\1.7_2.1.csv" # 数据路径
FEATURE_RESULTS_PATH = "feature_selection_results0420_0524.json" # 特征选择结果文件

TARGET_FEATURE_SET = "0.75_features"  # 指定使用.json文件中的哪个特征集
TARGET_FEATURE_SET2="0.5_features"
# ====================== 数据加载 ======================
def load_data():
    """加载数据（保持与原始代码一致的预处理）"""
    data = pd.read_csv(DATA_PATH)
   
    # 保持与原始代码完全一致的列定义
    continuous_cols = [
        'Age', 'Weight', 'WBC', 'RBC', 'Platelets', 'Hemoglobin', 
        'Glucose', 'Magnesium', 'Potassium', 'Sodium', 'Calcium',
        'Chloride', 'Phosphate', 'BUN',  'Creatinine','Charlson','sofa'
    ]
    
    categorical_cols = [
        'gender', 'diabetes','cerebrovascular_disease','congestive_heart_failure', 'renal_disease',
        'malignant_cancer','liver_disease','chronic_pulmonary_disease',
        'peripheral_vascular_disease', 'myocardial_infarct', 'hypertension','ventilation',
         'rrt', 'potassium_supplementation',
        'calcium_supplementation', 'magnesium_sulfate_supplementation'
    ]
    
    label_col = "death_ICU28days"
    
    X = data[continuous_cols + categorical_cols]
    y = data[label_col]
    
    # 保持与原始代码一致的分割方式
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=123
    )
    return X_train, X_test, y_train, y_test, continuous_cols, categorical_cols

# ====================== 独立评估函数 ======================
def standalone_evaluation():
    """独立评估流程"""
    # 加载数据
    X_train, X_test, y_train, y_test, cont_cols, cat_cols = load_data()
    
    # 加载特征选择结果
    with open(FEATURE_RESULTS_PATH) as f:
        feature_results = json.load(f)
    
    # 定义模型
    models = {
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=123),
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=123),
        'KNN': KNeighborsClassifier()
    }
    
    results = []
    
    for model_name in models:
        print(f"\n=== 正在评估 {model_name} ===")
        
        # 获取该模型的0.75特征集
        features = feature_results.get(model_name, {}).get(TARGET_FEATURE_SET, [])
        
        if not features:
            print(f"警告：{model_name} 无 {TARGET_FEATURE_SET}，跳过评估")
            continue
        
        # 创建预处理管道
        preprocessor = ColumnTransformer([
            ('num', StandardScaler(), [f for f in features if f in cont_cols]),
            ('cat', 'passthrough', [f for f in features if f in cat_cols])
        ])
        
        # 构建管道
        pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', models[model_name])
        ])
        
        # 训练与预测
        pipeline.fit(X_train[features], y_train)
        y_pred = pipeline.predict(X_test[features])
        y_proba = pipeline.predict_proba(X_test[features])[:,1]
        
        # 计算指标
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        metrics = {
            'AUC': round(roc_auc_score(y_test, y_proba), 3),
            'Sensitivity': round(tp/(tp+fn), 3),
            'Specificity': round(tn/(tn+fp), 3),
            'Accuracy': round((tp+tn)/(tp+tn+fp+fn), 3),
            'Num_Features': len(features)
        }
        
        results.append({
            'Model': model_name,
            **metrics,
            'Features': features
        })
    
    # 打印结果
    print("\n=== 最终评估结果 ===")
    result_df = pd.DataFrame(results).drop('Features', axis=1)
    print(result_df)
    
    # 保存详细结果
    pd.DataFrame(results).to_csv(f"独立评估结果0523_{TARGET_FEATURE_SET}.csv", index=False)
    print(f"\n详细结果已保存到 独立评估结果0523_{TARGET_FEATURE_SET}.csv")

# ====================== 执行 ======================
if __name__ == "__main__":
    standalone_evaluation()


=== 正在评估 RandomForest ===

=== 正在评估 LogisticRegression ===

=== 正在评估 KNN ===

=== 最终评估结果 ===
                Model    AUC  Sensitivity  Specificity  Accuracy  Num_Features
0        RandomForest  0.837        0.105        0.994     0.893            19
1  LogisticRegression  0.825        0.161        0.983     0.890             8
2                 KNN  0.713        0.121        0.985     0.887            20

详细结果已保存到 独立评估结果0523_0.75_features.csv
